Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import os

if not os.path.exists("data"):
    os.makedirs("data")

query = """
WITH t AS (
    SELECT
        CAST(get_json_object(dt.metadata, '$.correlation_id') AS STRING) AS chat_reference_id,
        rs.result,
        us.name AS annotatedBy,
        rs.annotated_at AS annotatedAt,
        dt.ticket_id AS ticket_id,
        to_timestamp(
            SUBSTRING(REPLACE(CAST(rs.annotated_at AS STRING), 'T', ' '), 1, 19),
            'yyyy-MM-dd HH:mm:ss'
        ) + INTERVAL 330 MINUTES AS annotated_at_ist
    FROM (
        SELECT
            id,
            CAST(data_id AS STRING) AS data_id,
            annotated_by,
            annotated_at,
            result,
            status
        FROM crm_ntl_annotation.annotation_results
        WHERE DATE(
            to_timestamp(
                SUBSTRING(REPLACE(CAST(annotated_at AS STRING), 'T', ' '), 1, 19),
                'yyyy-MM-dd HH:mm:ss'
            ) + INTERVAL 330 MINUTES
        ) >= DATE '2025-04-08'
    ) rs
    LEFT JOIN (
        SELECT
            CAST(id AS STRING) AS id,
            uuid,
            data,
            mime_type,
            batch_id,
            ticket_id,
            template_id,
            metadata,
            created_at,
            created_by,
            version,
            kafka_state,
            assignment_preferences,
            modified_at,
            modified_by,
            is_active,
            ticket_uuid
        FROM crm_ntl_annotation.annotation_data
    ) dt
        ON rs.data_id = dt.id
    LEFT JOIN mjolnir_service.users us
        ON us.account_id = rs.annotated_by
    WHERE dt.template_id = 177
      AND rs.result IS NOT NULL
      AND TRIM(rs.result) <> ''
      AND us.name IS NOT NULL
),

parsed AS (
    SELECT
        chat_reference_id,
        annotatedAt,
        annotatedBy,
        annotated_at_ist,
        result,
        from_json(
            CASE
                WHEN TRIM(result) LIKE '[%' THEN TRIM(result)
                ELSE CONCAT('[', TRIM(result), ']')
            END,
            'array<struct<
                from_name:string,
                value:struct<
                    choices:array<string>,
                    text:string
                >
            >>'
        ) AS x
    FROM t
),

flattened AS (
    SELECT
        p.chat_reference_id,
        p.annotatedBy,
        p.annotated_at_ist,
        item.from_name AS from_name,
        to_json(item.value.choices) AS choice_value,
        item.value.text AS text_value
    FROM parsed p
    LATERAL VIEW explode(p.x) item_table AS item
    WHERE p.x IS NOT NULL
      AND size(p.x) > 0
),

data AS (
    SELECT
        chat_reference_id,
        annotatedBy,
        DATE(annotated_at_ist) AS annotated_at_date,
        annotated_at_ist,

        MAX(CASE WHEN from_name = 'category' THEN choice_value END) AS Category,
        MAX(CASE WHEN from_name LIKE '%subcategory%' THEN choice_value END) AS `Sub-Category`,
        MAX(CASE WHEN from_name = 'correctint' THEN choice_value END) AS `Correct interpretation of customer's query / message`,
        MAX(CASE WHEN from_name = 'clarification' THEN choice_value END) AS `Asked for customer clarification for any response that it could not understand`,
        MAX(CASE WHEN from_name = 'Othertxn' THEN choice_value END) AS `Reference to correct LAN / transaction ID `,
        MAX(CASE WHEN from_name = 'correctinfo' THEN choice_value END) AS `Correct information as per SOP / guidelines`,
        MAX(CASE WHEN from_name = 'hallucination' THEN choice_value END) AS `No hallucinations (by GenAI Bot)`,
        MAX(CASE WHEN from_name = 'unnecessary' THEN choice_value END) AS `No Unnecessary information (answer not relevant to questions asked / entertaining random queries not related to Navi)`,
        MAX(CASE WHEN from_name = 'latch' THEN choice_value END) AS `Not latching to voice of customer (repeating VOC)`,
        MAX(CASE WHEN from_name = 'allqueries' THEN choice_value END) AS `Addressed all queries of the customer`,
        MAX(CASE WHEN from_name = 'allinfo' THEN choice_value END) AS `Given all information regarding a query(e.g. conveying 3-5 days TAT instead of saying soon)`,
        MAX(CASE WHEN from_name = 'empathy' THEN choice_value END) AS `Demonstrated understanding of customer emotions and concerns`,
        MAX(CASE WHEN from_name = 'easyunder' THEN choice_value END) AS `Replies are easy to understand (no technical terms / jargons)`,
        MAX(CASE WHEN from_name = 'language' THEN choice_value END) AS `Adherence to customer language preference`,
        MAX(CASE WHEN from_name = 'allmessage' THEN choice_value END) AS `Did GENAI consider all back-to back customer messages and not just last message?`,
        MAX(CASE WHEN from_name = 'loop' THEN choice_value END) AS `Did not go on a loop when customer said Thanks / Ok / any other acknowledgement`,
        MAX(CASE WHEN from_name = 'upload' THEN choice_value END) AS `Was it able to comprehend the uploaded document?`,
        MAX(CASE WHEN from_name = 'profess' THEN choice_value END) AS `Professional behaviour with customer(not referring to customers with slangs like bhai, yaar etc.)`,
        MAX(CASE WHEN from_name = 'openclose' THEN choice_value END) AS `Opening / Closing`,
        MAX(CASE WHEN from_name = 'cxend' THEN choice_value END) AS `Did customer end chat after we provided resolution?`,
        MAX(CASE WHEN from_name = 'rep' THEN choice_value END) AS `Unique responses by the agent (No word-to-word repititions)`,
        MAX(CASE WHEN from_name = 'closure' THEN choice_value END) AS `Did GENAI close the ticket? (If customer abandoned, select No)`,
        MAX(CASE WHEN from_name = 'csat' THEN choice_value END) AS `Closing script adherence (Correct CSAT script)`,
        MAX(CASE WHEN from_name = 'escalationagent' THEN choice_value END) AS `Correctly escalated to an agent`,
        MAX(CASE WHEN from_name = 'escalationnpci' THEN choice_value END) AS `Correctly escalated to NPCI`,
        MAX(CASE WHEN from_name = 'GENAI' THEN choice_value END) AS `Was GENAI Disposition correct?`,
        MAX(CASE WHEN from_name = 'remarks' THEN text_value END) AS Remarks
    FROM flattened
    WHERE DATE(annotated_at_ist) BETWEEN CURRENT_DATE() - INTERVAL 90 DAYS AND CURRENT_DATE()
    GROUP BY
        chat_reference_id,
        annotatedBy,
        annotated_at_ist
    HAVING COUNT(from_name) > 0
)

SELECT
    a.chat_reference_id,
    a.annotatedBy,
    a.annotated_at_date,
    a.annotated_at_ist,
    a.Category,
    a.`Sub-Category`,
    a.`Correct interpretation of customer's query / message`,
    a.`Asked for customer clarification for any response that it could not understand`,
    a.`Reference to correct LAN / transaction ID `,
    a.`Correct information as per SOP / guidelines`,
    a.`No hallucinations (by GenAI Bot)`,
    a.`No Unnecessary information (answer not relevant to questions asked / entertaining random queries not related to Navi)`,
    a.`Not latching to voice of customer (repeating VOC)`,
    a.`Addressed all queries of the customer`,
    a.`Given all information regarding a query(e.g. conveying 3-5 days TAT instead of saying soon)`,
    a.`Demonstrated understanding of customer emotions and concerns`,
    a.`Replies are easy to understand (no technical terms / jargons)`,
    a.`Adherence to customer language preference`,
    a.`Did GENAI consider all back-to back customer messages and not just last message?`,
    a.`Did not go on a loop when customer said Thanks / Ok / any other acknowledgement`,
    a.`Was it able to comprehend the uploaded document?`,
    a.`Professional behaviour with customer(not referring to customers with slangs like bhai, yaar etc.)`,
    a.`Opening / Closing`,
    a.`Did customer end chat after we provided resolution?`,
    a.`Unique responses by the agent (No word-to-word repititions)`,
    a.`Did GENAI close the ticket? (If customer abandoned, select No)`,
    a.`Closing script adherence (Correct CSAT script)`,
    a.`Correctly escalated to an agent`,
    a.`Correctly escalated to NPCI`,
    a.`Was GENAI Disposition correct?`,
    a.Remarks,
    b.csat
FROM data a
LEFT JOIN (
    SELECT
        CAST(canonical_ticket_id AS STRING) AS canonical_ticket_id,
        AVG(csat) AS csat
    FROM corp_analytics.cx_sub_ticket_data_model_tbl
    GROUP BY CAST(canonical_ticket_id AS STRING)
) b
    ON a.chat_reference_id = b.canonical_ticket_id
WHERE COALESCE(
    a.Category,
    a.`Sub-Category`,
    a.`Correct interpretation of customer's query / message`,
    a.`Correct information as per SOP / guidelines`,
    a.Remarks
) IS NOT NULL
ORDER BY a.annotated_at_date DESC
"""

df = get_sql_df(query)

def make_unique_columns(columns):
    seen = {}
    unique_cols = []
    for col in columns:
        if col not in seen:
            seen[col] = 0
            unique_cols.append(col)
        else:
            seen[col] += 1
            unique_cols.append(f"{col} ({seen[col]})")
    return unique_cols

df.columns = make_unique_columns(list(df.columns))

gsheet_url = "https://docs.google.com/spreadsheets/d/1AQRa2wh45WRtd3XLnxn91HYh4QFq4_Ll7GafOk5HhL8/edit?gid=0#gid=0"
worksheet_name = "UPI-GenAI"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(worksheet_name, start_row=1, start_column=1)
gs_reader.set_df_in_sheets(worksheet_name, df, include_headers=True, row=1, col=1)